In [ ]:
import shutil
import os
import matplotlib.pyplot as plt
import matplotlib.image as img
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array


def augment_images(input_dir, output_dir, target_count, datagen):
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    
    existing_files = os.listdir(input_dir)
    existing_count = len(existing_files)
   
    # 复制原始图像到输出目录（避免重复）
    for filename in existing_files:
        src_path = os.path.join(input_dir, filename)
        dst_path = os.path.join(output_dir, filename)
        if not os.path.exists(dst_path):  # 避免覆盖已有文件
            shutil.copy(src_path, dst_path) #将原始图像从 src_path复制到 dst_path
    
    # 计算需要生成的增强图像数量
    current_count = len(os.listdir(output_dir))  # 当前总数量
    copies_needed = (target_count - current_count 
                     + len(existing_files) - 1) // current_count  
    
    if copies_needed==0:
        return
    #  生成增强图像
    for filename in existing_files:
        img_path = os.path.join(input_dir, filename)
        img = Image.open(img_path) #使用PIL库，打开当前特定图片
        img_array = np.array(img)

        # 注意Keras的输入要求 (height, width, channels),区别于Pytorch的输入
        if len(img_array.shape) == 2:  # 灰度图像
            img_array = np.expand_dims(img_array, axis=-1)
            img_array = np.repeat(img_array, 3, axis=-1)  # 转换为3通道
        
        img_array = np.expand_dims(img_array, axis=0)  # 添加batch维度
        # 生成增强图像直到满足目标数量
        # 对当前原始图片生成多次增强版本，比如3200/64=50次
        i = 0
        for batch in datagen.flow(img_array, batch_size=1):
            if current_count >= target_count:
                break
                
            augmented_img = batch[0].astype('uint8')
            if augmented_img.shape[-1] == 3:  # 转换回灰度
                augmented_img = augmented_img.mean(axis=-1).astype('uint8')
            
            save_path = os.path.join(output_dir, f"aug_{i}_{filename}")
            Image.fromarray(augmented_img).save(save_path)
            current_count += 1
            i += 1
            
            if i >= copies_needed:
                break

In [ ]:
# 定义不同类别的增强配置
moderate_datagen = ImageDataGenerator(
    rotation_range=45, # 随机旋转角度范围(±45度)
    width_shift_range=0.3,#水平平移范围(30%宽度)
    height_shift_range=0.3,
    shear_range=0.2, # 剪切变换强度
    zoom_range=0.2,# 随机缩放范围(80%-120%)
    horizontal_flip=True,# 允许水平翻转
    fill_mode='nearest',# 填充新像素的方式(最近邻)
    brightness_range=[0.6, 1.4]  # 亮度调整
)

mild_datagen = ImageDataGenerator(
    rotation_range=30, 
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

very_mild_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

non_demented_datagen = ImageDataGenerator()  # 基本不增强

In [ ]:
'''对数据集图像不足的部分做数据增强处理'''

# 原始数据路径
base_dir = '../input/alzheimer-mri-4-classes-dataset/Alzheimer_MRI_4_classes_dataset'
baseout_dir = './augmented_data'  
os.makedirs(baseout_dir, exist_ok=True)


# 分别增强四类数据
augment_images(
    input_dir=os.path.join(base_dir, 'ModerateDemented'),
    output_dir=os.path.join(baseout_dir, 'ModerateDemented'),
    target_count=3200,
    datagen=moderate_datagen
)

augment_images(
    input_dir=os.path.join(base_dir, 'MildDemented'),
    output_dir=os.path.join(baseout_dir, 'MildDemented'),
    target_count=3200,
    datagen=mild_datagen
)

augment_images(
    input_dir=os.path.join(base_dir, 'VeryMildDemented'),
    output_dir=os.path.join(baseout_dir, 'VeryMildDemented'),
    target_count=3200,
    datagen=very_mild_datagen
)

augment_images(
    input_dir=os.path.join(base_dir, 'NonDemented'),
    output_dir=os.path.join(baseout_dir, 'NonDemented'),
    target_count=3200,
    datagen=non_demented_datagen
)

print("Data augmentation completed!")

In [ ]:
# 生成增强后的数据集
print("正在压缩数据集...")
!cd /kaggle/working
!zip -qr augmented_data.zip augmented_data
print("压缩完成！文件保存为 augmented_data.zip")